# dYdX v4 Markets Discovery

Explore all available perpetual markets on dYdX v4 chain.

## About dYdX
- **Type**: Cosmos-based blockchain (dYdX v4 launched 2023)
- **Decentralized**: Fully on-chain order book and settlement
- **Available**: Globally (no KYC, no geo-restrictions)
- **Settlement**: USDC (cross-margined)
- **Funding**: Hourly (charged at XX:00:00 UTC)

## Funding Rate Formula
- **Funding Rate = (Premium Component / 8) + Interest Rate**
- Premium: TWAP of mark-index spread over 1 hour
- Interest Rate: 0% for cross markets, 0.125 bps/hour for isolated
- Charged **every hour** (not every 8 hours!)

## API Details
- **Mainnet**: https://indexer.dydx.trade/v4
- **Market Format**: BTC-USD, ETH-USD (hyphenated)
- **Timezone**: UTC

In [ ]:
import requests
import pandas as pd
from datetime import datetime

BASE_URL = "https://indexer.dydx.trade/v4"

## Check Server Time and Timezone

In [ ]:
# Get server time
response = requests.get(f"{BASE_URL}/time")
server_time = response.json()
print(f"Server time: {server_time}")

# Get current height
response = requests.get(f"{BASE_URL}/height")
height_data = response.json()
print(f"\nCurrent block: {height_data}")

## Discover All Perpetual Markets

In [ ]:
# Get all perpetual markets
response = requests.get(f"{BASE_URL}/perpetualMarkets")
markets_data = response.json()

print(f"Total markets: {len(markets_data['markets'])}")
print(f"\nFirst market example:")
first_market = list(markets_data['markets'].values())[0]
for key, value in first_market.items():
    print(f"  {key}: {value}")

## Parse Market Data

In [ ]:
markets = []
for ticker, market in markets_data['markets'].items():
    markets.append({
        'ticker': ticker,
        'base_asset': market.get('baseAsset', ''),
        'quote_asset': market.get('quoteAsset', ''),
        'price': float(market.get('oraclePrice', 0)),
        'volume_24h': float(market.get('volume24H', 0)),
        'open_interest': float(market.get('openInterestUSDC', 0)) / 1e6,  # Convert to USDC
        'trades_24h': int(market.get('trades24H', 0)),
        'next_funding_rate': float(market.get('nextFundingRate', 0)),
        'status': market.get('status', ''),
    })

df_markets = pd.DataFrame(markets)
df_markets = df_markets.sort_values('volume_24h', ascending=False)

print(f"Total markets: {len(df_markets)}")
print(f"\nMarket status distribution:")
print(df_markets['status'].value_counts())

## Top Markets by Volume

In [ ]:
# Show top 50 markets by volume
top_markets = df_markets.head(50)

print("Top 50 Markets by 24h Volume:")
print("=" * 100)
for idx, row in top_markets.iterrows():
    print(f"{row['ticker']:<15} | Vol: ${row['volume_24h']/1e6:>8.1f}M | OI: ${row['open_interest']:>8.1f}M | "
    f"Price: ${row['price']:>10.2f} | Funding: {row['next_funding_rate']*100:>6.4f}%")

In [ ]:
top_markets